In [1]:
import os
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from tqdm import tqdm
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import GridSearchCV
import xgboost as xgb
import time

In [2]:
# Load the dataset split files
train_df = pd.read_csv(r"datasets\DAIC-Woz\train_split_Depression_AVEC2017.csv")
dev_df = pd.read_csv(r"datasets\DAIC-Woz\dev_split_Depression_AVEC2017.csv")

In [3]:
def load_audio_features(participant_id, data_path):
    participant_id_str = str(int(participant_id))  # Convert to string and remove decimal point
    covarep_file = os.path.join(data_path, f'{participant_id_str}_P', f'{participant_id_str}_COVAREP.csv')
    formant_file = os.path.join(data_path, f'{participant_id_str}_P', f'{participant_id_str}_FORMANT.csv')
    
    covarep_data = pd.read_csv(covarep_file)
    formant_data = pd.read_csv(formant_file)
    
    covarep_mean = covarep_data.mean(axis=0)
    formant_mean = formant_data.mean(axis=0)
    
    audio_features = pd.concat([covarep_mean, formant_mean], axis=0)
    return audio_features

In [4]:

# Modify the load_all_features function to include audio features
def load_all_features(participant_id, data_path):
    audio_features = load_audio_features(participant_id, data_path)
    
    # Concatenate all features into a single feature vector
    all_features = pd.concat([audio_features], axis=0)
    return all_features


In [5]:
# Path to the dataset
data_path = r"datasets\DAIC-Woz"

In [6]:
# Extract features and labels for the training set
train_features = []
train_labels = []

In [7]:
for idx, row in tqdm(train_df.iterrows(), total=train_df.shape[0]):
    participant_id = int(row["Participant_ID"])
    label = int(row["PHQ8_Binary"])
    try:
        features = load_all_features(participant_id, data_path)
        train_features.append(features)
        train_labels.append(label)
    except FileNotFoundError:
        print(f"File not found for participant {participant_id}")


100%|██████████| 107/107 [04:41<00:00,  2.63s/it]


In [8]:
train_features = np.array(train_features)
train_labels = np.array(train_labels)

In [12]:
print(train_features)
print(train_labels)

[[2.22958961e+02 5.07327866e-01 8.43676125e-02 ... 2.56033944e+03
  3.47597315e+03 4.36572803e+03]
 [2.38712935e+02 4.09581246e-01 7.48307346e-02 ... 2.53810169e+03
  3.46428480e+03 4.37248558e+03]
 [1.34162521e+02 3.82220553e-01 5.15591725e-02 ... 2.48986906e+03
  3.50464837e+03 4.41917830e+03]
 ...
 [2.14761104e+02 4.60723009e-01 7.69773914e-02 ... 2.57704765e+03
  3.45149344e+03 4.27562785e+03]
 [2.30017697e+02 3.56899085e-01 7.45908010e-02 ... 2.51137498e+03
  3.41340551e+03 4.33214651e+03]
 [1.94475615e+02 4.09452302e-01 6.58954265e-02 ... 2.52902217e+03
  3.45477522e+03 4.34516514e+03]]
[0 0 0 0 0 0 0 0 0 0 1 1 1 0 0 1 0 0 0 1 0 0 1 1 0 0 0 1 1 1 1 1 1 1 1 1 1
 0 0 0 1 0 0 0 0 0 0 0 1 0 0 1 0 1 0 0 1 0 0 0 0 0 0 1 0 1 1 0 0 0 0 0 1 0
 0 0 0 1 0 0 1 0 0 0 0 0 1 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0]


In [13]:
# Train-Test Split
X_train, X_val, y_train, y_val = train_test_split(train_features, train_labels, test_size=0.2, random_state=42)

# Apply SMOTE to balance the dataset
#smote = SMOTE(random_state=42)
#X_resampled, y_resampled = smote.fit_resample(X_train, y_train)


In [14]:
# Initialize the classifier
xgb_clf = xgb.XGBClassifier(random_state=42)


In [15]:
# Fit the model
xgb_clf.fit(X_train, y_train)

XGBoostError: [08:42:12] C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-06abd128ca6c1688d-1\xgboost\xgboost-ci-windows\src\common\../data/gradient_index.h:94: Check failed: valid: Input data contains `inf` or a value too large, while `missing` is not set to `inf`

In [ ]:
# Predictions
y_pred = xgb_clf.predict(X_val)

# Evaluate the model
accuracy = accuracy_score(y_val, y_pred)
report = classification_report(y_val, y_pred)

print(f'Accuracy: {accuracy}')
print(f'Classification Report:\n{report}')

In [1]:
import os
#from pyAudioAnalysis import audioSegmentation as aS
#from pyAudioAnalysis.audioBasicIO import read_audio_file, stereo_to_mono
from pydub import AudioSegment
from pydub.silence import split_on_silence
import librosa
import numpy as np
#from pyAudioAnalysis import audioFeatureExtraction
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score

c:\Users\sachl\Desktop\Depression detection\.venv\Lib\site-packages\pydub\utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)


In [2]:
# Function to extract features from audio using Librosa
def extract_features_librosa(audio_file):
    y, sr = librosa.load(audio_file, sr=16000)  # Load audio file at 16kHz sampling rate
    y, _ = librosa.effects.trim(y)  # Trim leading and trailing silence

    # Extract Mel Frequency Cepstral Coefficients (MFCCs)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    mfcc_mean = np.mean(mfcc, axis=1)

    # Compute other features using librosa if needed (e.g., spectral centroid, zero crossing rate, etc.)

    return mfcc_mean

In [3]:
# Function to preprocess audio
def preprocess_audio(audio_file):
    # Load audio using Librosa
    y, sr = librosa.load(audio_file, sr=16000)  # Load audio file at 16kHz sampling rate
    
    # Trim leading and trailing silence using librosa
    y, _ = librosa.effects.trim(y)
    
    # Split audio into segments based on silence
    segments = split_on_silence(AudioSegment.from_file(audio_file), min_silence_len=500, silence_thresh=-30)

    # Remove segments where computed energy falls below a certain threshold
    segments_to_keep = [segment for segment in segments if segment.dBFS > -50]
    
    # Concatenate segments back into one audio file
    output_audio = AudioSegment.empty()
    for segment in segments_to_keep:
        output_audio += segment

    # Save the cleaned audio file
    output_file = 'cleaned_audio.wav'
    output_audio.export(output_file, format="wav")

    return output_file

In [ ]:
# Function to extract features from audio using pyAudioAnalysis
#def extract_features_pyaudioanalysis(audio_file):
#    [features, _] = audioFeatureExtraction.stFeatureExtraction(audio_file, 16000, 16000, 0.050*16000, 0.025*16000)
#    features_mean = np.mean(features, axis=1)
#    return features_mean

In [4]:
dir_path = r'datasets\DAIC-Woz'
#participants = range(300, 493)  # Assuming participants range from 300 to 492


audio_compiled = []
#y = []

for folder_name in os.listdir(dir_path):
    folder_path = os.path.join(dir_path, folder_name)

    if os.path.isdir(folder_path) and folder_name.endswith("_P"):
        participant_id = int(folder_name.split("_")[0])

        audio_file = os.path.join(dir_path, f'{participant_id}_AUDIO.wav')

        if os.path.exists(audio_file):
            mfcc_features = extract_features_librosa(audio_file)

            


#for participant in participants:
#
    
#label = 0 
    
# Extract features using Librosa
#mfcc_features = extract_features_librosa(audio_file)
    
# Alternatively, extract features using pyAudioAnalysis
# features = extract_features_pyaudioanalysis(audio_file)
    
#X.append(mfcc_features)
#y.append(label)

# Convert lists to numpy arrays
#X = np.array(X)
#y = np.array(y)

In [5]:
display(X)

NameError: name 'X' is not defined

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Train SVM model
model = SVC(kernel='linear', random_state=42)
model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
# Evaluate model
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

print(f'Accuracy: {accuracy}')
print('Classification Report:')
print(report)